# Air Pollution Prediction - Data Exploration

This notebook provides an interactive exploration of air quality data.

## Contents
1. Data Loading
2. Data Overview
3. Statistical Analysis
4. Visualizations
5. Correlation Analysis
6. Temporal Patterns

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
from src.data_loader import load_data, create_sample_data
from src.preprocessing import preprocess_data
from src.visualization import (
    plot_time_series, 
    plot_correlation_matrix,
    plot_aqi_distribution
)
from config import POLLUTION_FEATURES, METEOROLOGICAL_FEATURES

# Configure plotting
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries imported successfully!")

## 1. Data Loading

In [ ]:
# Select city
city = "Delhi"  # Change to Mumbai, Chennai, or Bangalore

# Load data
print(f"Loading data for {city}...")
df = load_data(city)

print(f"\n✅ Loaded {len(df)} records")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")

## 2. Data Overview

In [ ]:
# Display first few rows
print("First 10 rows:")
df.head(10)

In [ ]:
# Data info
print("Dataset Information:")
df.info()

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
missing[missing > 0]

## 3. Statistical Analysis

In [ ]:
# Statistical summary
print("Statistical Summary:")
df[POLLUTION_FEATURES + METEOROLOGICAL_FEATURES + ['AQI']].describe()

In [ ]:
# AQI category distribution
def get_aqi_category(aqi):
    if aqi <= 50:
        return 'Good'
    elif aqi <= 100:
        return 'Satisfactory'
    elif aqi <= 200:
        return 'Moderate'
    elif aqi <= 300:
        return 'Poor'
    elif aqi <= 400:
        return 'Very Poor'
    else:
        return 'Severe'

df['AQI_Category'] = df['AQI'].apply(get_aqi_category)

print("\nAQI Category Distribution:")
print(df['AQI_Category'].value_counts())
print("\nPercentage:")
print(df['AQI_Category'].value_counts(normalize=True) * 100)

## 4. Visualizations

In [ ]:
# AQI distribution
plot_aqi_distribution(df['AQI'].values, title=f"AQI Distribution - {city}")

In [ ]:
# Time series - AQI
plot_time_series(df, ['AQI'], title=f"AQI Over Time - {city}")

In [ ]:
# Time series - Pollution parameters
plot_time_series(df, ['PM2.5', 'PM10'], title=f"PM Levels Over Time - {city}")

In [ ]:
# Pollution distribution
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(POLLUTION_FEATURES):
    axes[i].hist(df[feature], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{feature} Distribution', fontweight='bold')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation heatmap
plot_correlation_matrix(
    df, 
    columns=POLLUTION_FEATURES + METEOROLOGICAL_FEATURES + ['AQI'],
    title=f"Feature Correlation Matrix - {city}"
)

In [ ]:
# Correlation with AQI
corr_with_aqi = df[POLLUTION_FEATURES + METEOROLOGICAL_FEATURES].corrwith(df['AQI']).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
corr_with_aqi.plot(kind='barh', color='teal')
plt.title('Correlation with AQI', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nCorrelation with AQI:")
print(corr_with_aqi)

## 6. Temporal Patterns

In [ ]:
# Extract temporal features
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Quarter'] = df['Date'].dt.quarter

# Monthly average AQI
monthly_aqi = df.groupby('Month')['AQI'].mean()

plt.figure(figsize=(12, 5))
monthly_aqi.plot(kind='bar', color='coral')
plt.title('Average AQI by Month', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Average AQI')
plt.xticks(range(12), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Day of week pattern
dow_aqi = df.groupby('DayOfWeek')['AQI'].mean()

plt.figure(figsize=(10, 5))
dow_aqi.plot(kind='bar', color='steelblue')
plt.title('Average AQI by Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Average AQI')
plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], rotation=0)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Quarterly trends
quarterly_stats = df.groupby('Quarter')[['PM2.5', 'PM10', 'AQI']].mean()

quarterly_stats.plot(kind='bar', figsize=(10, 6))
plt.title('Quarterly Pollution Trends', fontsize=14, fontweight='bold')
plt.xlabel('Quarter')
plt.ylabel('Average Value')
plt.xticks(rotation=0)
plt.legend(['PM2.5', 'PM10', 'AQI'])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nQuarterly Statistics:")
print(quarterly_stats)

## 7. Key Insights

### Summary:
- **Dataset Size:** [X] records
- **Average AQI:** [X.XX]
- **Most Polluted Month:** [Month]
- **Most Correlated Features:** PM2.5, PM10
- **Seasonal Pattern:** Higher pollution in winter months

### Next Steps:
1. Preprocess data (handle missing values, outliers)
2. Engineer features (lag features, rolling statistics)
3. Train machine learning models
4. Evaluate and compare models
5. Generate predictions

In [ ]:
# Save exploration results
print("\n" + "="*60)
print("EXPLORATION COMPLETE!")
print("="*60)
print(f"\nDataset: {city}")
print(f"Records: {len(df)}")
print(f"Features: {len(df.columns)}")
print(f"Average AQI: {df['AQI'].mean():.2f}")
print(f"Max AQI: {df['AQI'].max():.2f}")
print(f"Min AQI: {df['AQI'].min():.2f}")
print("\nProceed to the next notebook: 02_model_training.ipynb")